In [8]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report
import m2cgen as m2c

In [9]:
# ==========================================
# 1. LOAD DATA
# ==========================================
print("--- Loading detect_dataset.csv ---")
try:
    df = pd.read_csv('./dataset/detect_dataset.csv', usecols=['Ia', 'Va', 'Output (S)'])
except FileNotFoundError:
    raise FileNotFoundError("File 'detect_dataset.csv' not found. Make sure the dataset is placed in the ./dataset/ folder.")

print("Raw dataset shape:", df.shape)
print(df.head())

--- Loading detect_dataset.csv ---
Raw dataset shape: (12001, 3)
   Output (S)          Ia        Va
0           0 -170.472196  0.054490
1           0 -122.235754  0.102000
2           0  -90.161474  0.141026
3           0  -79.904916  0.156272
4           0  -63.885255  0.180451


In [10]:
df

,Output (S),Ia,Va
0,0,-170.472196,0.054490
1,0,-122.235754,0.102000
2,0,-90.161474,0.141026
3,0,-79.904916,0.156272
4,0,-63.885255,0.180451
...,...,...,...
11996,0,-66.237921,0.094421
11997,0,-65.849493,0.103778
11998,0,-65.446698,0.113107
11999,0,-65.029633,0.122404


In [11]:
# ==========================================
# 2. SINGLE-PHASE EXTRACTION (Phase A only)
# ==========================================
# The sensor measures a single phase (V and I on the same line).
# We use phase A (Va, Ia) as the training data.
#
# Why NOT stack all three phases:
#   The fault label Output(S)=1 is system-level — it means a fault exists somewhere
#   in the 3-phase system, NOT necessarily on the phase being observed.
#   For asymmetric faults (e.g. a fault on phase B only), Va and Ia remain normal
#   while the label is still 1. Stacking would inject (Va, Ia, Fault=1) samples
#   where V and I look normal → label noise that degrades model quality.
#
# Why using one phase is physically correct:
#   A single-phase sensor can only detect anomalies it directly measures.
#   If a fault is isolated to phase B, the sensor on phase A cannot see it —
#   so the model should NOT flag it. Using only phase A keeps labels consistent
#   with what the sensor physically observes.
#
# SIGN HANDLING — abs() transform:
#   The dataset contains signed instantaneous AC values (oscillating +/-).
#   IoT sensors (current transformer + ADC, voltage divider + ADC) always output
#   positive magnitudes. Applying abs() maps the training data to the same
#   always-positive space, so the model's learned thresholds match sensor readings.
df_1phase = pd.DataFrame({
    'V': df['Va'].abs(),
    'I': df['Ia'].abs(),
    'Fault': df['Output (S)']
})

print("First 5 rows (absolute values):")
print(df_1phase.head())
print("\nClass distribution:")
print(df_1phase['Fault'].value_counts())

First 5 rows (absolute values):
          V           I  Fault
0  0.054490  170.472196      0
1  0.102000  122.235754      0
2  0.141026   90.161474      0
3  0.156272   79.904916      0
4  0.180451   63.885255      0

Class distribution:
Fault
0    6505
1    5496
Name: count, dtype: int64


In [12]:
# ==========================================
# 3. SPLIT DATASET & TRAIN MODEL (TINYML)
# ==========================================
X = df_1phase[['V', 'I']]
y = df_1phase['Fault']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Define a Decision Tree classifier
dt = DecisionTreeClassifier(random_state=42)

# CONSTRAIN TREE SIZE:
# max_depth is capped at 3 so the generated C code stays small enough
# to fit in the RAM of a Contiki-NG node.
param_grid = {
    'max_depth': [2, 3],
    'min_samples_split': [5, 10, 20]
}

print("--- Training model with GridSearchCV ---")
grid_search = GridSearchCV(estimator=dt, param_grid=param_grid, cv=5, scoring='f1_macro')
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print(f"Best compact tree parameters: {grid_search.best_params_}")

# Evaluate the compact model on the test set
y_pred = best_model.predict(X_test)
print("\n--- Classification Report on Test Set ---")
print(classification_report(y_test, y_pred))

--- Training model with GridSearchCV ---
Best compact tree parameters: {'max_depth': 3, 'min_samples_split': 5}

--- Classification Report on Test Set ---
              precision    recall  f1-score   support

           0       0.85      0.99      0.92      1301
           1       0.98      0.80      0.88      1100

    accuracy                           0.90      2401
   macro avg       0.92      0.90      0.90      2401
weighted avg       0.91      0.90      0.90      2401



In [16]:
import os

# ==========================================
# 4. EXPORT MODEL TO C HEADER & DEPLOY TO SENSOR
# ==========================================
# The generated header is placed directly in the shared sensor directory
# so all three sensor types (wind, solar, diesel) can #include it
# without any Makefile changes — they all compile coap-mqtt-sensor.c.
SENSOR_DIR = "../sensors/coap-mqtt-sensor"
OUTPUT_FILE = os.path.join(SENSOR_DIR, "detect_anomaly.h")

# m2cgen exports a DecisionTreeClassifier as:
#   void score(double * input, double * output)
# where output[] holds per-class probabilities: output[0]=P(normal), output[1]=P(fault)
# We keep this internal function and wrap it in a clean sensor API:
#   int detect_anomaly(float v, float i)  →  1=fault, 0=normal
c_code = m2c.export_to_c(best_model)

# Downcast double -> float to reduce memory footprint on the MCU
c_code = c_code.replace("double", "float")

# Rename the internal scorer so it doesn't conflict with the public API
c_code = c_code.replace("void score(", "static void _anomaly_score(")

# Build the public wrapper:
#   - takes scalar v and i directly (matches sensor measurement variables)
#   - passes them as an array to the generated scorer
#   - returns 1 if P(fault) > 0.5, else 0
wrapper = (
    "\n"
    "/* Public API — call this from the sensor firmware */\n"
    "/* v = absolute single-phase voltage, i = absolute single-phase current */\n"
    "/* returns: 1 = anomaly detected, 0 = normal                            */\n"
    "static int detect_anomaly(float v, float i) {\n"
    "    float input[2] = {v, i};\n"
    "    float output[2];\n"
    "    _anomaly_score(input, output);\n"
    "    return output[1] > 0.5f ? 1 : 0;\n"
    "}\n"
)

header = (
    "#ifndef DETECT_ANOMALY_H\n"
    "#define DETECT_ANOMALY_H\n\n"
    "/* Auto-generated by tinyML notebook — DO NOT EDIT MANUALLY */\n"
    "/* Model: DecisionTreeClassifier trained on single-phase (V, I) fault data */\n\n"
    + c_code
    + wrapper
    + "\n#endif /* DETECT_ANOMALY_H */\n"
)

os.makedirs(SENSOR_DIR, exist_ok=True)
with open(OUTPUT_FILE, "w") as f:
    f.write(header)

print(f"Model exported to: {os.path.abspath(OUTPUT_FILE)}")
print("\n--- File contents ---")
print(header)

Model exported to: /home/khaido1/IoT/IoT_project/sensors/coap-mqtt-sensor/detect_anomaly.h

--- File contents ---
#ifndef DETECT_ANOMALY_H
#define DETECT_ANOMALY_H

/* Auto-generated by tinyML notebook — DO NOT EDIT MANUALLY */
/* Model: DecisionTreeClassifier trained on single-phase (V, I) fault data */

#include <string.h>
static void _anomaly_score(float * input, float * output) {
    float var0[2];
    if (input[1] <= 72.90320587158203) {
        if (input[0] <= 0.5086576044559479) {
            if (input[1] <= 19.780384063720703) {
                memcpy(var0, (float[]){0.15139442231075698, 0.848605577689243}, 2 * sizeof(float));
            } else {
                memcpy(var0, (float[]){0.812484076433121, 0.187515923566879}, 2 * sizeof(float));
            }
        } else {
            if (input[0] <= 0.5185004472732544) {
                memcpy(var0, (float[]){0.9727272727272728, 0.02727272727272727}, 2 * sizeof(float));
            } else {
                memcpy(var0, (float

In [14]:
# ==========================================
# 5. EXAMPLE (V, I) VALUES FROM THE MODEL
# ==========================================
from sklearn.tree import export_text

# Print the decision tree rules so we can see the learned thresholds
print("=== Decision Tree Rules ===")
print(export_text(best_model, feature_names=['V', 'I']))

# Pull real examples from the test set, grouped by prediction
X_test_copy = X_test.copy()
X_test_copy['True_Fault'] = y_test.values
X_test_copy['Predicted'] = y_pred

normal_examples   = X_test_copy[X_test_copy['Predicted'] == 0].head(5)
fault_examples    = X_test_copy[X_test_copy['Predicted'] == 1].head(5)

print("\n--- Examples predicted as NORMAL (Fault=0) ---")
print(normal_examples.to_string(index=False))

print("\n--- Examples predicted as FAULT (Fault=1) ---")
print(fault_examples.to_string(index=False))

# Manual probe using positive values only — matching what the sensor actually sends
print("\n--- Manual probe (positive values, as sensor would report) ---")
probe_values = [
    (0.0,    0.0),
    (0.5,   50.0),
    (0.5,  200.0),
    (1.0,  100.0),
    (1.0,  300.0),
]
print(f"{'V':>8}  {'I':>8}  {'Prediction':>12}")
print("-" * 32)
for v, i in probe_values:
    pred = best_model.predict([[v, i]])[0]
    label = "FAULT" if pred == 1 else "normal"
    print(f"{v:>8.3f}  {i:>8.3f}  {label:>12}")

=== Decision Tree Rules ===
|--- I <= 72.90
|   |--- V <= 0.51
|   |   |--- I <= 19.78
|   |   |   |--- class: 1
|   |   |--- I >  19.78
|   |   |   |--- class: 0
|   |--- V >  0.51
|   |   |--- V <= 0.52
|   |   |   |--- class: 0
|   |   |--- V >  0.52
|   |   |   |--- class: 0
|--- I >  72.90
|   |--- I <= 83.66
|   |   |--- V <= 0.07
|   |   |   |--- class: 1
|   |   |--- V >  0.07
|   |   |   |--- class: 0
|   |--- I >  83.66
|   |   |--- I <= 90.39
|   |   |   |--- class: 1
|   |   |--- I >  90.39
|   |   |   |--- class: 1


--- Examples predicted as NORMAL (Fault=0) ---
       V         I  True_Fault  Predicted
0.178741 60.338310           0          0
0.587566  0.801728           0          0
0.512862 21.538257           0          0
0.353016 54.474201           0          0
0.609634 15.941685           0          0

--- Examples predicted as FAULT (Fault=1) ---
       V          I  True_Fault  Predicted
0.170592 836.684385           1          1
0.040451 179.840371           1 

/home/khaido1/IoT/IoT_project/env/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/home/khaido1/IoT/IoT_project/env/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/home/khaido1/IoT/IoT_project/env/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/home/khaido1/IoT/IoT_project/env/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/home/khaido1/IoT/IoT_project/env/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does